In [ ]:
# !pip install langchain-text-splitters

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [16]:
def chunking_function(fp, chunks_function):
    with open(fp, 'r', encoding='utf-8') as file:
        text = file.read()
        chunks = chunks_function(text)
        embeddings = model.encode(chunks)
    return chunks, embeddings

## Different chunking

### 1. Fixed length chunking

In [17]:
def fixed_length_chunking(text, chunk_size=512):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [18]:
# with open("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", 'r') as file:
#     text = file.read()
# Fc_chunks = fixed_length_chunking(text, chunk_size=512)
# Fc_chunk_embeddings = model.encode(Fc_chunks)
Fc_chunks, Fc_chunk_embeddings = chunking_function("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", fixed_length_chunking)

In [20]:
Fc_chunks[0]

'1. Introduction\nAt our organization, we firmly believe that our employees are the cornerstone of our success. As a leading retail company operating across stores, warehouses, and corporate offices, we are committed to fostering a work environment that promotes not only professional growth but also personal well-being.\nThis Employee Benefits Policy has been designed to provide a comprehensive overview of the various benefits available to employees. It aims to ensure fairness, transparency, and consistency in'

### 2. Context Aware Chunking

In [ ]:
def content_aware_chunking(text):
    paragraphs = text.split("\n\n")  # split by paragraph
    chunks = [p.strip() for p in paragraphs if p.strip()]
    return chunks

In [21]:
# with open("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", 'r') as file:
#     text = file.read()
# cac_chunks = content_aware_chunking(text)
# cac_chunk_embeddings = model.encode(cac_chunks)
cac_chunks, cac_chunk_embeddings = chunking_function("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", content_aware_chunking)

In [22]:
cac_chunks[0]

'1. Introduction\nAt our organization, we firmly believe that our employees are the cornerstone of our success. As a leading retail company operating across stores, warehouses, and corporate offices, we are committed to fostering a work environment that promotes not only professional growth but also personal well-being.\nThis Employee Benefits Policy has been designed to provide a comprehensive overview of the various benefits available to employees. It aims to ensure fairness, transparency, and consistency in how benefits are administered, while also aligning with statutory requirements and industry best practices within India.\nThe provisions outlined in this document are intended to support employees through different stages of their professional and personal lives, thereby enabling them to perform at their best.'

### 3. Recursive Character Chunking

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def recursive_chunking(text, chunk_size=512, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=["\n\n", "\n", " ", ""])
    chunks = text_splitter.split_text(text)
    return chunks

In [37]:
rc_chunks, rc_chunk_embeddings = chunking_function("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", recursive_chunking)

### 4. Document Structure-Based Chunking

In [43]:
import re

def structure_based_chunking(text):
    pattern = r"\nChapter |\nSection |\n\d+\.\s|\n"
    sections = re.split( pattern,text)  # split by headings
    chunks = []
    
    for sec in sections:
        sec = sec.strip()
        if sec:
            chunks.append(sec)
    
    return chunks

In [44]:
DSC_chunks, DSC_chunk_embeddings = chunking_function("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", structure_based_chunking)

### 5. Semantic Chunking

In [47]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

def semantic_chunk(text, model_name='all-MiniLM-L6-v2', percentile_threshold=85):
    model = SentenceTransformer(model_name)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) < 2: # Can't find a break if there's only 1 sentence
        return [text]
    embeddings = model.encode(sentences)
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        similarities.append(sim)
    threshold = np.percentile(similarities, 100 - percentile_threshold)
    breakpoints = [i for i, sim in enumerate(similarities) if sim < threshold]
    chunks = []
    start_idx = 0
    for bp in breakpoints:
        # bp is the index of the *last* sentence in the chunk
        chunk = ' '.join(sentences[start_idx : bp + 1])
        chunks.append(chunk)
        start_idx = bp + 1 # next chunk starts after the break
    chunks.append(' '.join(sentences[start_idx:]))

    return chunks

In [48]:
sc_chunks, sc_chunk_embeddings = chunking_function("D:\\LLM-Learning-Journey\\LLM-Learning-Journey\\Week3 - Embedding and vector Database\\Data\\HR_Employee_benefit.txt", semantic_chunk)

### Contextual Chunking